# Day 36：阶段项目二(上)：企业级知识库问答系统——架构、多格式入库、混合检索与FastAPI服务（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 6 周**：第6周：阶段项目二——企业级知识库问答系统实战与答辩  
> **GitLab 工单**：`STARMIND-036` ｜ **开发分支**：`feat/day36-project2-kb-backend-api`  
> **配套 HTML5 课件**：`courseware/day36/index.html` ｜ **独立版本快照**：`versions/day36/`

---

## 🎙️ 老陈开场旁白

各位同学，激动人心的时刻到了！今天正式启动第三阶段收官大战——**阶段项目二：星辰智脑企业级知识库问答系统（StarMind-KB v2.0）**！
产品经理林姐已经把正式 PRD 签发下来了，明天下午就要跟前端联调，后天（Day 38）就要上台做技术答辩与代码评审！
咱们按正规大厂两日冲刺节奏拆解：
- **今天（项目二·上）**：打牢后端地基！完成**多格式文档解析入库（Markdown / TXT / JSON / CSV）**、**部门级权限过滤（ACL Pre-Filter）**、**BM25 + 向量 RRF 混合检索 + Cross-Encoder 精排引擎**，并用 **FastAPI** 暴露出带 Pydantic 严格校验的标准 RESTful API（文档入库、文档列表、文档删除、问答查询、健康检查），再用 `TestClient` 跑通全接口自动化集成测试！
别再写脚本级玩具了，今天写的每一行代码都要按「直接能部署到测试环境」的工业标准来！

### 承上启下
- **昨日基线**：前两周（Day 25~35）我们把 LangChain、文档加载切分、向量检索、高级 RAG 混合检索+Rerank、Ragas 评估和 LlamaIndex 核心原理全部打通了。现在，是时候把所有零散积木拼装成一座真正可上线的企业级系统了！
- **明日预告**：明天（Day 37）我们将完成**阶段项目二（下）**：为 StarMind-KB v2.0 补齐多会话记忆、多轮指代消解改写、引用溯源 `<mark>` 高亮展示、SSE 打字机流式输出接口与全链路 TestClient 联调！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day36'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-036）

杭州星辰数科全员 800+ 员工急需统一的内部企业知识库问答平台（StarMind-KB v2.0），覆盖 HR 制度、财务报销、研发规范与运维排障四类多格式文档，并要求严格的部门数据权限隔离与可溯源回答。

**今日迭代目标：**
- 构建 `MultiFormatParser`：统一解析 `md`、`txt`、`json`、`csv` 四种企业常见文档格式并切分保留 `section / department / chunk_index` 元数据。
- 构建 `EnterpriseKBEngine`：支持文档幂等 Upsert、按 `doc_id` 软/硬删除、部门 ACL 权限过滤、`BM25 + Dense RRF + Cross-Encoder Rerank` 精排与低分防幻觉拒答。
- 基于 FastAPI 构建生产级 REST API：`GET /health`、`POST /api/v1/documents/ingest`、`GET /api/v1/documents`、`DELETE /api/v1/documents/{doc_id}`、`POST /api/v1/qa/query`。

**验收标准：**
- 成功通过 HTTP API 分别入库 Markdown、JSON、CSV 文档，并支持相同 `doc_id` 重复入库时自动覆盖旧切片（不产生脏数据）。
- 当 `department='rd'` 发起查询时，绝不返回 `department='finance'` 的机密文档切片（ACL 隔离 100% 生效）。
- 针对库外无关提问（如「火星大气成分」）自动触发防幻觉拒答（`refused=True`）。
- 使用 `fastapi.testclient.TestClient` 完成全接口集成测试，100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么企业知识库必须做「ACL 预过滤（Pre-Filtering）」？

如果先不带权限全库检索 Top-3，再把其中 2 条机密文档删掉，最终只剩 1 条甚至 0 条结果——这叫 **Post-Filtering 召回坍缩**！正确做法是在计算 BM25 / 向量相似度之前，先按 `department in (user_dept, 'public')` 锁定合法候选文档池，既 100% 杜绝越权泄露，又保证合法结果永远填满 Top-K。

> **⚠️ 老陈避坑**：踩坑提醒：用户所属部门绝对不能由前端随手传一个未鉴权的字段说了算！本节课为演示清晰直接在请求体传 `department`，真实生产环境必须从服务端验签后的 JWT Token / SSO Session 里解析用户真实角色与部门。

### 2. 2. CSV 表格类文档切分防失真秘籍

千万不要把 CSV 直接当纯文本按字符数硬切！一旦把表头切在第 1 块、第 20 行数据切在第 4 块，第 4 块只剩 `杭州,450,100`，大模型根本不知道 450 是住宿费还是餐补！像本节课 `MultiFormatParser` 那样，**把每一行数据与表头动态拼成 `城市: 杭州；住宿上限: 450；餐补: 100`**，检索精度瞬间起飞。

> **⚠️ 老陈避坑**：踩坑提醒：超大 Excel/CSV（上万行）不应全部塞进向量库，应转为 SQLite/PostgreSQL 数据库，走第四阶段的 Text-to-SQL Agent 工具查询！


## 💻 实操步骤 1：步骤一：实现多格式文档解析、ACL 权限过滤与混合精排引擎（EnterpriseKBEngine）

- **目标文件**：`versions/day36/src/day36_kb_engine.py`
- **老陈代码旁白**：咱们先把项目二的核心领域引擎写扎实：`MultiFormatParser` 处理 MD/TXT/JSON/CSV 四种企业文档，`EnterpriseKBEngine` 负责幂等 Upsert、文档删除、部门 ACL 隔离、BM25+向量 RRF 混合检索、Cross-Encoder 精排与低分防幻觉拒答。


In [ ]:
"""
Day 36 - 步骤一：阶段项目二(上) 企业级知识库核心引擎
支持：多格式文档解析(MD/TXT/JSON/CSV)、doc_id 幂等 Upsert/删除、部门 ACL 过滤、
     BM25 + Dense 向量 RRF 混合检索 + Cross-Encoder 精排 + 阈值拒答
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import re
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple


@dataclass
class KBChunk:
    chunk_id: str
    doc_id: str
    source: str
    section: str
    department: str
    content: str
    chunk_index: int


@dataclass
class CitationItem:
    doc_id: str
    source: str
    section: str
    snippet: str
    score: float


@dataclass
class QAResult:
    question: str
    answer: str
    citations: List[CitationItem]
    refused: bool
    latency_ms: float
    used_rerank: bool


def tokenize_text(text: str) -> List[str]:
    lowered = text.lower()
    tokens = re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", lowered)
    for chunk in re.findall(r"[\u4e00-\u9fff]+", lowered):
        for ch in chunk:
            tokens.append(ch)
        for i in range(len(chunk) - 1):
            tokens.append(chunk[i : i + 2])
    return tokens


class MultiFormatParser:
    """企业多格式文档统一解析与语义切分器（支持 md / txt / json / csv）。"""

    @staticmethod
    def parse(
        doc_id: str,
        source: str,
        fmt: str,
        content: str,
        department: str = "public",
        chunk_size: int = 180,
    ) -> List[KBChunk]:
        fmt_lower = fmt.lower().strip(".")
        chunks: List[KBChunk] = []

        if fmt_lower == "md":
            # 按 Markdown 二级标题分段保留章节名
            sections = re.split(r"(?m)^##\s+", content.strip())
            idx = 0
            for raw_sec in sections:
                raw_sec = raw_sec.strip()
                if not raw_sec:
                    continue
                lines = raw_sec.splitlines()
                sec_title = lines[0].strip("# ") if len(lines) > 1 else "正文"
                body = "\n".join(lines[1:]).strip() if len(lines) > 1 else raw_sec
                for start in range(0, max(1, len(body)), chunk_size):
                    piece = body[start : start + chunk_size].strip()
                    if not piece:
                        continue
                    chunks.append(
                        KBChunk(
                            chunk_id=f"{doc_id}#c{idx}",
                            doc_id=doc_id,
                            source=source,
                            section=sec_title,
                            department=department,
                            content=piece,
                            chunk_index=idx,
                        )
                    )
                    idx += 1
            return chunks

        if fmt_lower == "csv":
            reader = csv.DictReader(io.StringIO(content.strip()))
            for idx, row in enumerate(reader):
                kv_text = "；".join(f"{k.strip()}: {str(v).strip()}" for k, v in row.items() if k)
                if kv_text:
                    chunks.append(
                        KBChunk(
                            chunk_id=f"{doc_id}#c{idx}",
                            doc_id=doc_id,
                            source=source,
                            section=f"表格第{idx + 1}行",
                            department=department,
                            content=kv_text,
                            chunk_index=idx,
                        )
                    )
            return chunks

        if fmt_lower == "json":
            data = json.loads(content)
            items = data if isinstance(data, list) else [data]
            for idx, item in enumerate(items):
                if isinstance(item, dict):
                    sec = str(item.get("code") or item.get("title") or f"条目{idx + 1}")
                    text = "；".join(f"{k}: {v}" for k, v in item.items())
                else:
                    sec = f"条目{idx + 1}"
                    text = str(item)
                chunks.append(
                    KBChunk(
                        chunk_id=f"{doc_id}#c{idx}",
                        doc_id=doc_id,
                        source=source,
                        section=sec,
                        department=department,
                        content=text,
                        chunk_index=idx,
                    )
                )
            return chunks

        # 默认按 txt 段落切分
        paras = [p.strip() for p in re.split(r"\n{2,}", content.strip()) if p.strip()]
        for idx, p in enumerate(paras):
            chunks.append(
                KBChunk(
                    chunk_id=f"{doc_id}#c{idx}",
                    doc_id=doc_id,
                    source=source,
                    section=f"段落{idx + 1}",
                    department=department,
                    content=p,
                    chunk_index=idx,
                )
            )
        return chunks


class EnterpriseKBEngine:
    """StarMind-KB v2.0 核心引擎：多格式入库、ACL过滤、混合RRF检索与Cross-Encoder精排。"""

    def __init__(self, refuse_threshold: float = 0.25, dim: int = 96) -> None:
        self.refuse_threshold = refuse_threshold
        self.dim = dim
        self.doc_registry: Dict[str, Dict[str, Any]] = {}
        self.chunks: List[KBChunk] = []
        self._chunk_tfs: Dict[str, Counter[str]] = {}
        self._chunk_vecs: Dict[str, List[float]] = {}

    def _embed(self, text: str) -> List[float]:
        vec = [0.0] * self.dim
        for tok in tokenize_text(text):
            h = int(hashlib.md5(tok.encode("utf-8")).hexdigest(), 16)
            vec[h % self.dim] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        return [v / norm for v in vec]

    def ingest_document(
        self,
        doc_id: str,
        source: str,
        fmt: str,
        content: str,
        department: str = "public",
    ) -> int:
        """幂等入库：若 doc_id 已存在则先清除旧切片再写入新版本。"""
        self.delete_document(doc_id)
        new_chunks = MultiFormatParser.parse(
            doc_id=doc_id, source=source, fmt=fmt, content=content, department=department
        )
        for ch in new_chunks:
            self.chunks.append(ch)
            self._chunk_tfs[ch.chunk_id] = Counter(tokenize_text(ch.content + " " + ch.section))
            self._chunk_vecs[ch.chunk_id] = self._embed(ch.content + " " + ch.section)

        self.doc_registry[doc_id] = {
            "doc_id": doc_id,
            "source": source,
            "format": fmt,
            "department": department,
            "chunk_count": len(new_chunks),
        }
        return len(new_chunks)

    def delete_document(self, doc_id: str) -> bool:
        if doc_id not in self.doc_registry:
            return False
        kept: List[KBChunk] = []
        for ch in self.chunks:
            if ch.doc_id == doc_id:
                self._chunk_tfs.pop(ch.chunk_id, None)
                self._chunk_vecs.pop(ch.chunk_id, None)
            else:
                kept.append(ch)
        self.chunks = kept
        self.doc_registry.pop(doc_id, None)
        return True

    def list_documents(self) -> List[Dict[str, Any]]:
        return list(self.doc_registry.values())

    def _rerank_score(self, query: str, chunk: KBChunk) -> float:
        q_tokens = set(tokenize_text(query))
        if not q_tokens:
            return 0.0
        c_tokens = set(self._chunk_tfs[chunk.chunk_id].keys())
        q_bigrams = {t for t in q_tokens if len(t) >= 2}
        c_bigrams = {t for t in c_tokens if len(t) >= 2}
        overlap_bi = len(q_bigrams & c_bigrams) / max(1, len(q_bigrams)) if q_bigrams else 0.0
        overlap_all = len(q_tokens & c_tokens) / max(1, len(q_tokens))
        q_codes = set(re.findall(r"[a-z0-9]+[-_][a-z0-9]+", query.lower()))
        code_bonus = 0.35 if (q_codes and any(c in chunk.content.lower() for c in q_codes)) else 0.0
        return round(min(1.0, 0.55 * overlap_bi + 0.35 * overlap_all + code_bonus), 4)

    def search(
        self,
        question: str,
        top_k: int = 3,
        department: str = "public",
        use_rerank: bool = True,
    ) -> List[Tuple[KBChunk, float]]:
        # 1. ACL 权限预过滤（Pre-Filtering）
        allowed_depts = {"public", department}
        pool = [ch for ch in self.chunks if ch.department in allowed_depts]
        if not pool:
            return []

        q_tokens = tokenize_text(question)
        q_vec = self._embed(question)

        # 2. BM25 稀疏打分
        n_docs = len(pool)
        df: Counter[str] = Counter()
        lengths: Dict[str, int] = {}
        for ch in pool:
            tf = self._chunk_tfs[ch.chunk_id]
            lengths[ch.chunk_id] = sum(tf.values())
            for t in tf.keys():
                df[t] += 1
        avgdl = sum(lengths.values()) / max(1, n_docs)

        bm25_ranked: List[Tuple[KBChunk, float]] = []
        for ch in pool:
            tf = self._chunk_tfs[ch.chunk_id]
            dl = lengths[ch.chunk_id]
            s = 0.0
            for t in q_tokens:
                freq = tf.get(t, 0)
                if freq == 0:
                    continue
                idf = math.log(1.0 + (n_docs - df[t] + 0.5) / (df[t] + 0.5))
                s += idf * (freq * 2.5) / (freq + 1.5 * (0.25 + 0.75 * dl / max(1e-9, avgdl)))
            if s > 0:
                bm25_ranked.append((ch, s))
        bm25_ranked.sort(key=lambda x: x[1], reverse=True)

        # 3. Dense 向量余弦打分
        dense_ranked: List[Tuple[KBChunk, float]] = []
        for ch in pool:
            c_vec = self._chunk_vecs[ch.chunk_id]
            sim = sum(x * y for x, y in zip(q_vec, c_vec))
            if sim > 0:
                dense_ranked.append((ch, sim))
        dense_ranked.sort(key=lambda x: x[1], reverse=True)

        # 4. RRF 倒数排名融合
        rrf_scores: Dict[str, float] = {}
        chunk_map: Dict[str, KBChunk] = {}
        for rank, (ch, _) in enumerate(bm25_ranked[:10], start=1):
            chunk_map[ch.chunk_id] = ch
            rrf_scores[ch.chunk_id] = rrf_scores.get(ch.chunk_id, 0.0) + 0.5 / (60 + rank)
        for rank, (ch, _) in enumerate(dense_ranked[:10], start=1):
            chunk_map[ch.chunk_id] = ch
            rrf_scores[ch.chunk_id] = rrf_scores.get(ch.chunk_id, 0.0) + 0.5 / (60 + rank)

        candidates = [
            chunk_map[cid]
            for cid, _ in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:8]
        ]
        if not candidates:
            return []

        # 5. Cross-Encoder 精排（或直接返回归一化特征分）
        scored = [(ch, self._rerank_score(question, ch)) for ch in candidates]
        if use_rerank:
            scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def query(
        self,
        question: str,
        top_k: int = 3,
        department: str = "public",
        use_rerank: bool = True,
    ) -> QAResult:
        t0 = time.perf_counter()
        hits = self.search(question, top_k=top_k, department=department, use_rerank=use_rerank)
        latency_ms = round((time.perf_counter() - t0) * 1000.0, 2)

        if not hits or hits[0][1] < self.refuse_threshold:
            return QAResult(
                question=question,
                answer="抱歉，星辰知识库当前授权范围内未检索到足够可靠的依据，无法回答该问题。",
                citations=[],
                refused=True,
                latency_ms=latency_ms,
                used_rerank=use_rerank,
            )

        citations = [
            CitationItem(
                doc_id=ch.doc_id,
                source=ch.source,
                section=ch.section,
                snippet=ch.content[:140],
                score=sc,
            )
            for ch, sc in hits
            if sc >= self.refuse_threshold
        ]
        evidence_lines = [
            f"[{i}] {c.snippet}（来源：{c.source} · {c.section}）"
            for i, c in enumerate(citations, start=1)
        ]
        answer = f"根据星辰数科知识库检索结果，针对「{question}」解答如下：\n" + "\n".join(evidence_lines)
        return QAResult(
            question=question,
            answer=answer,
            citations=citations,
            refused=False,
            latency_ms=latency_ms,
            used_rerank=use_rerank,
        )


def build_seeded_kb_engine() -> EnterpriseKBEngine:
    """预置星辰数科多格式演示知识库（MD + CSV + JSON）。"""
    engine = EnterpriseKBEngine()
    engine.ingest_document(
        doc_id="hr-handbook-2026",
        source="员工休假与差旅手册.md",
        fmt="md",
        department="public",
        content=(
            "## 年假与病假规定\n"
            "入职满1年员工享有5天带薪年假，年假需提前3个工作日提交OA审批；病假超过2天须提供公立医院诊断证明。\n\n"
            "## 生产发布窗口\n"
            "业务代码日常发布窗口为周一至周四10:00-18:00，严禁周五17:00后提交非紧急上线变更。"
        ),
    )
    engine.ingest_document(
        doc_id="fin-travel-table",
        source="差旅住宿标准表.csv",
        fmt="csv",
        department="public",
        content="城市等级,代表城市,住宿上限,餐饮补助\n一线城市,北京/上海/深圳/广州,每晚600元,每天100元\n新一线城市,杭州/成都/武汉,每晚450元,每天100元\n",
    )
    engine.ingest_document(
        doc_id="ops-err-dict",
        source="网关错误码字典.json",
        fmt="json",
        department="rd",
        content=json.dumps(
            [
                {"code": "ERR-5042", "meaning": "统一网关调用上游推理服务超时", "action": "检查上游Pod显存并调大proxy_read_timeout"},
                {"code": "ERR-4019", "meaning": "零信任VPN动态令牌校验失败", "action": "同步客户端NTP时钟"},
            ],
            ensure_ascii=False,
        ),
    )
    engine.ingest_document(
        doc_id="fin-salary-secret",
        source="2026高管薪酬方案.txt",
        fmt="txt",
        department="finance",
        content="2026年高管年终奖金池基数为税前净利润的8%，仅限财务总监与CEO查阅。",
    )
    return engine


if __name__ == "__main__":
    kb = build_seeded_kb_engine()
    print("已入库文档:", [(d["doc_id"], d["department"], d["chunk_count"]) for d in kb.list_documents()])
    r1 = kb.query("ERR-5042 怎么处理？", department="rd")
    print("\n研发查错误码:", r1.answer)
    r2 = kb.query("高管年终奖金池基数是多少？", department="rd")
    print("\n研发越权查高管薪酬 refused=", r2.refused)


### 🔍 步骤 1 代码深度解析

1. `MultiFormatParser` 把 CSV 的每一行与表头拼成语义陈述句，把 JSON 错误码字典转为带 `code` 章节名的独立切片，完美适配企业异构数据源。
2. `EnterpriseKBEngine.search` 在最前面执行 `pool = [ch for ch in self.chunks if ch.department in {'public', department}]`，确保研发人员搜「高管年终奖金池」直接触发 `refused=True`，滴水不漏！

**预期运行输出参考：**
```text
已入库文档: [('hr-handbook-2026', 'public', 2), ('fin-travel-table', 'public', 2), ('ops-err-dict', 'rd', 2), ('fin-salary-secret', 'finance', 1)]

研发查错误码: 根据星辰数科知识库检索结果...

研发越权查高管薪酬 refused= True
```


## 💻 实操步骤 2：步骤二：构建 StarMind-KB v2.0 FastAPI 后端服务与 Pydantic 强类型契约

- **目标文件**：`versions/day36/src/day36_kb_api.py`
- **老陈代码旁白**：引擎就位，现在用 FastAPI 把它包装成标准微服务！注意我们用 `create_app(engine=None)` 应用工厂模式，加上完整的 Pydantic 请求/响应校验和清晰的 HTTP 状态码。


In [ ]:
"""
Day 36 - 步骤二：阶段项目二(上) StarMind-KB v2.0 FastAPI 后端服务
提供：健康检查、多格式文档入库(Upsert)、文档列表/删除、混合检索+Rerank 问答 API
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day36_kb_engine import EnterpriseKBEngine, build_seeded_kb_engine


class IngestRequest(BaseModel):
    doc_id: str = Field(..., min_length=2, max_length=64, description="文档唯一ID（同ID重复入库自动覆盖）")
    source: str = Field(..., min_length=1, description="文档文件名")
    format: str = Field(..., pattern="^(md|txt|json|csv)$", description="文档格式：md/txt/json/csv")
    content: str = Field(..., min_length=1, description="文档原始内容")
    department: str = Field(default="public", description="可见部门：public/rd/finance/hr")


class IngestResponse(BaseModel):
    doc_id: str
    chunks_created: int
    status: str = "ingested"


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=2, max_length=500)
    top_k: int = Field(default=3, ge=1, le=10)
    department: str = Field(default="public")
    use_rerank: bool = Field(default=True)


class CitationDTO(BaseModel):
    doc_id: str
    source: str
    section: str
    snippet: str
    score: float


class QueryResponse(BaseModel):
    question: str
    answer: str
    citations: List[CitationDTO]
    refused: bool
    latency_ms: float
    used_rerank: bool


def create_app(engine: Optional[EnterpriseKBEngine] = None) -> FastAPI:
    """FastAPI 应用工厂：支持生产默认预置知识库，也支持测试注入独立隔离引擎。"""
    kb = engine if engine is not None else build_seeded_kb_engine()
    app = FastAPI(
        title="StarMind-KB v2.0 企业级知识库问答系统",
        version="2.0.0",
        description="杭州星辰数科企业知识库：多格式入库 + ACL权限过滤 + BM25/向量混合检索 + Rerank",
    )

    @app.get("/health")
    def health() -> Dict[str, Any]:
        return {
            "status": "ok",
            "doc_count": len(kb.doc_registry),
            "chunk_count": len(kb.chunks),
        }

    @app.post("/api/v1/documents/ingest", response_model=IngestResponse)
    def ingest_document(req: IngestRequest) -> IngestResponse:
        try:
            count = kb.ingest_document(
                doc_id=req.doc_id,
                source=req.source,
                fmt=req.format,
                content=req.content,
                department=req.department,
            )
        except Exception as exc:
            raise HTTPException(status_code=400, detail=f"文档解析失败: {exc}") from exc
        return IngestResponse(doc_id=req.doc_id, chunks_created=count)

    @app.get("/api/v1/documents")
    def list_documents() -> Dict[str, Any]:
        docs = kb.list_documents()
        return {"total": len(docs), "items": docs}

    @app.delete("/api/v1/documents/{doc_id}")
    def delete_document(doc_id: str) -> Dict[str, Any]:
        deleted = kb.delete_document(doc_id)
        if not deleted:
            raise HTTPException(status_code=404, detail=f"文档 {doc_id} 不存在")
        return {"deleted": True, "doc_id": doc_id}

    @app.post("/api/v1/qa/query", response_model=QueryResponse)
    def qa_query(req: QueryRequest) -> QueryResponse:
        res = kb.query(
            question=req.question,
            top_k=req.top_k,
            department=req.department,
            use_rerank=req.use_rerank,
        )
        return QueryResponse(
            question=res.question,
            answer=res.answer,
            citations=[
                CitationDTO(
                    doc_id=c.doc_id,
                    source=c.source,
                    section=c.section,
                    snippet=c.snippet,
                    score=c.score,
                )
                for c in res.citations
            ],
            refused=res.refused,
            latency_ms=res.latency_ms,
            used_rerank=res.used_rerank,
        )

    return app


app = create_app()

if __name__ == "__main__":
    from fastapi.testclient import TestClient

    client = TestClient(create_app())
    print("GET /health ->", client.get("/health").json())
    resp = client.post("/api/v1/qa/query", json={"question": "北京出差住宿上限是多少？", "department": "public"})
    print("POST /api/v1/qa/query ->", resp.json()["answer"])


### 🔍 步骤 2 代码深度解析

1. `IngestRequest` 使用 `pattern='^(md|txt|json|csv)$'` 在接口入口直接拦截不支持的文件类型，无需污染核心引擎。
2. `create_app()` 返回独立 FastAPI 实例，既可以用 `uvicorn src.day36_kb_api:app --reload` 启动真实服务，也能被 `TestClient` 毫秒级跑通全链路测试！

**预期运行输出参考：**
```text
GET /health -> {'status': 'ok', 'doc_count': 4, 'chunk_count': 7}
POST /api/v1/qa/query -> 根据星辰数科知识库检索结果，针对「北京出差住宿上限是多少？」解答如下：
[1] 城市等级: 一线城市；代表城市: 北京/上海/深圳/广州；住宿上限: 每晚600元；餐饮补助: 每天100元（来源：差旅住宿标准表.csv · 表格第1行）
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day36/tests/test_day36.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 36 自动化集成测试：使用 FastAPI TestClient 验证多格式入库、ACL权限隔离、问答与删除。"""

from fastapi.testclient import TestClient

from src.day36_kb_api import create_app


def test_health_and_seeded_qa_with_acl():
    client = TestClient(create_app())

    h = client.get("/health")
    assert h.status_code == 200
    assert h.json()["status"] == "ok"
    assert h.json()["doc_count"] == 4

    # 1. 公共 CSV 表格问答
    r_pub = client.post(
        "/api/v1/qa/query",
        json={"question": "一线城市北京出差住宿上限是多少？", "department": "public"},
    )
    assert r_pub.status_code == 200
    body = r_pub.json()
    assert body["refused"] is False
    assert "600元" in body["answer"]
    assert len(body["citations"]) >= 1

    # 2. 部门 ACL 隔离：研发人员不可查财务机密，财务人员可查
    r_rd_denied = client.post(
        "/api/v1/qa/query",
        json={"question": "2026年高管年终奖金池基数是多少？", "department": "rd"},
    )
    assert r_rd_denied.json()["refused"] is True

    r_fin_ok = client.post(
        "/api/v1/qa/query",
        json={"question": "2026年高管年终奖金池基数是多少？", "department": "finance"},
    )
    assert r_fin_ok.json()["refused"] is False
    assert "8%" in r_fin_ok.json()["answer"]


def test_ingest_upsert_and_delete_lifecycle():
    client = TestClient(create_app())

    # 入库一篇新 Markdown 文档
    r1 = client.post(
        "/api/v1/documents/ingest",
        json={
            "doc_id": "it-wifi-01",
            "source": "办公网指南.md",
            "format": "md",
            "department": "public",
            "content": "## 访客WiFi\n访客WiFi名称为 StarMind-Guest，每日密码需在行政钉钉群获取。",
        },
    )
    assert r1.status_code == 200
    assert r1.json()["chunks_created"] == 1

    # 同 doc_id 幂等更新覆盖旧内容
    r2 = client.post(
        "/api/v1/documents/ingest",
        json={
            "doc_id": "it-wifi-01",
            "source": "办公网指南_v2.md",
            "format": "md",
            "department": "public",
            "content": "## 访客WiFi\n访客WiFi已升级为 StarMind-Guest-5G，支持扫码一键认证。",
        },
    )
    assert r2.status_code == 200

    q = client.post("/api/v1/qa/query", json={"question": "访客WiFi怎么连接？", "department": "public"})
    assert "StarMind-Guest-5G" in q.json()["answer"]
    assert "钉钉群" not in q.json()["answer"]

    # 删除文档后应无法再次删除（404）
    d1 = client.delete("/api/v1/documents/it-wifi-01")
    assert d1.status_code == 200
    d2 = client.delete("/api/v1/documents/it-wifi-01")
    assert d2.status_code == 404


def test_out_of_domain_query_triggers_refusal():
    client = TestClient(create_app())
    r = client.post(
        "/api/v1/qa/query",
        json={"question": "火星大气层主要成分是什么？", "department": "public"},
    )
    assert r.status_code == 200
    assert r.json()["refused"] is True
    assert r.json()["citations"] == []

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：为多格式解析器增加「YAML 配置文件」格式支持

**题目要求：**
编写 `parse_simple_yaml_kv(text)`，将简单的 `key: value` 行式 YAML 配置解析为自然语言陈述切片列表，并验证对运维服务配置文档的解析结果。

**思路提示：** 跳过空行与 `#` 注释行，按首个 `:` 拆分键值并格式化为 `配置项 {k} 的值为 {v}`。


In [ ]:
"""Day 36 作业一参考答案：YAML 键值配置文档解析器"""
from typing import List


def parse_simple_yaml_kv(text: str) -> List[str]:
    chunks: List[str] = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        k, v = line.split(":", 1)
        chunks.append(f"配置项 {k.strip()} 的标准值为 {v.strip()}")
    return chunks


if __name__ == "__main__":
    yml = "# 网关配置\nproxy_read_timeout: 30s\nmax_retries: 3\n"
    out = parse_simple_yaml_kv(yml)
    print(out)
    assert len(out) == 2 and "30s" in out[0]


**答案解析与评分标准：**

研发运维知识库中大量存在 YAML/INI 配置说明，结构化转为自然语言陈述可显著提升检索命中率。


### 作业 2（企业实战）：作业二：实现基于文档内容 SHA-256 的重复上传短路跳过（Deduplication）

**题目要求：**
维护一个 `doc_hashes: Dict[str, str]`，实现 `should_skip_ingest(doc_hashes, doc_id, content)`：若同一 `doc_id` 再次上传且内容哈希完全一致，返回 `True`（跳过重复切分与向量化省算力）；若内容有变动则更新哈希并返回 `False`。

**思路提示：** 使用 `hashlib.sha256(content.encode('utf-8')).hexdigest()` 计算内容指纹。


In [ ]:
"""Day 36 作业二参考答案：SHA-256 文档入库幂等去重器"""
import hashlib
from typing import Dict


def should_skip_ingest(doc_hashes: Dict[str, str], doc_id: str, content: str) -> bool:
    digest = hashlib.sha256(content.encode("utf-8")).hexdigest()
    if doc_hashes.get(doc_id) == digest:
        return True
    doc_hashes[doc_id] = digest
    return False


if __name__ == "__main__":
    hashes: Dict[str, str] = {}
    assert should_skip_ingest(hashes, "d1", "版本1内容") is False
    assert should_skip_ingest(hashes, "d1", "版本1内容") is True   # 内容未变，短路跳过
    assert should_skip_ingest(hashes, "d1", "版本2内容") is False  # 内容更新，触发重建
    print("SHA-256 增量去重验证通过!")


**答案解析与评分标准：**

定时同步 Confluence/飞书知识库时，95% 的文档每天都没改动；加一层 SHA-256 指纹比对能直接省掉 95% 的 Embedding 算力费用。


### 作业 3（架构进阶）：作业三：实现多部门联合角色权限校验（RBAC 多标签过滤）

**题目要求：**
一名跨部门管理者可能同时拥有 `['public', 'rd', 'finance']` 多个权限标签，而一篇敏感文档可能要求 `required_roles=['finance', 'exec']`（满足任一即可见）。请实现 `filter_authorized_chunks(chunks, user_roles)` 并验证权限交集判断。

**思路提示：** 使用 Python 集合交集 `bool(set(user_roles) & set(chunk_roles))` 判断是否放行。


In [ ]:
"""Day 36 作业三参考答案：RBAC 多角色权限交集过滤器"""
from typing import Dict, List, Sequence, Set


def filter_authorized_chunks(
    chunks: Sequence[Dict],
    user_roles: Sequence[str],
) -> List[Dict]:
    u_set: Set[str] = set(user_roles) | {"public"}
    return [ch for ch in chunks if u_set & set(ch.get("allowed_roles", ["public"]))]


if __name__ == "__main__":
    docs = [
        {"id": "c1", "allowed_roles": ["public"]},
        {"id": "c2", "allowed_roles": ["rd", "ops"]},
        {"id": "c3", "allowed_roles": ["finance", "exec"]},
    ]
    rd_only = filter_authorized_chunks(docs, ["rd"])
    vp_roles = filter_authorized_chunks(docs, ["rd", "exec"])
    print("普通研发可见:", [d["id"] for d in rd_only])
    print("技术VP可见:", [d["id"] for d in vp_roles])
    assert [d["id"] for d in rd_only] == ["c1", "c2"]
    assert [d["id"] for d in vp_roles] == ["c1", "c2", "c3"]


**答案解析与评分标准：**

真实大型企业的文档权限很少是单字符串部门，升级为集合交集的 RBAC 模型即可无缝对接企业微信/飞书/LDAP 组织架构。


---

## 🎯 老陈收尾复盘

非常稳！StarMind-KB v2.0 的后端大动脉今天已经彻底打通：多格式入库、增量更新、安全删除、部门 ACL 隔离、混合检索+Rerank 精排、防幻觉拒答、FastAPI 标准接口一气呵成！明天（Day 37）咱们给它装上「多轮对话大脑 + 引用高亮 + SSE 流式打字机」交互闭环！
